Here we do an initial combination of the Open Air Chicago clarity sensors and satellite retrieved NO2 tropospheric column measurements.
Hourly and spatial aggregations on the open air data are performed. Basic statistic metrics are calculated. 
This notebook does not currently filter out pixels with single sites, but that is planned as a future addition per request of Carl Malings. 

- Authorship: Noah Ring
- Year: 2026
- Program: EarthRISE Developers Academy
- Location: Langley Research Center

In [2]:
import xarray as xr
import matplotlib as plt
import pandas as pd
import geopandas as gpd

# Change the matplotlib dpi to increase or decrease output size in vscode
plt.rcParams['figure.dpi'] = 150 

# Change some of the default xarray settings
xr.set_options(keep_attrs=True, display_expand_data=False)

# EPSG to be used when setting coordinate reference systems
epsg = 4326 

# Function used for coercing clarity lat/lon to match the 0.2 x 0.2deg grid of TEMPO
# Note: I included the bias term (0.01) to force the lat/lon to match TEMPO lat/lon
def round_002(x):
    out = 0.02*round(x/0.02) + 0.01
    return out


In [3]:

#Define the path where the tempo.nc file is stored.
tempo_path = "~/Documents/01_Research/larc-chicago-haq/datasets/tempo.nc"

# Define the path where the open air chicago clarity data is stored
clarity_path = "~/Documents/01_Research/larc-chicago-haq/datasets/Open_Air_Chicago_Individual_Measurements_20260615.csv"

# Define path where the shapefiles for chicago are stored.
# Note: This is currently unused in this file.
community_shapefiles = (
                        "~/Documents/01_Research/larc-chicago-haq/shapefiles/"
                        "ChicagoCommunityAreasBoundaries/geo_export_6383d95e-1e67-4fe2-8f45-534d73a53a92.shp"
                        )

# Use xarray to load tempo data
tempo_raster = xr.open_dataset(tempo_path, decode_coords=True, decode_times=True)
tempo_raster = tempo_raster.rename({"vertical_column_troposphere":"column_no2"})
tempo_raster

# Load in community boundaries
chi_communities = gpd.read_file(community_shapefiles)
# chi_communities = chi_communities.set_crs(epsg="EPSG:4326")

# Load clarity data .csv
clarity_full_dat = pd.read_csv(clarity_path)

# Subset the clarity sensor data for our variables of interest
# Reformat the datetime variable, and add other time columns to possibly be used later
clarity_subset = clarity_full_dat[['datasourceid', 'sensor_name', 'latitude', 'longitude', 'time',
                            'no2ConcIndividual.value']]
clarity_subset['datetime'] = pd.to_datetime(clarity_subset["time"], format="%m/%d/%Y %I:%M:%S %p")
clarity_subset['month'] = clarity_subset['datetime'].dt.month
clarity_subset["year"] = clarity_subset['datetime'].dt.year
clarity_subset["month_year_str"] = clarity_subset['month'].astype(str) + "-" + clarity_subset['year'].astype(str)
clarity_subset["month_year"] = pd.to_datetime(clarity_subset["month_year_str"], format="%m-%Y")

# Convert clarity subset to geopandas dataframe
clarity_subset["geometry"] = gpd.points_from_xy(
    x = clarity_subset["longitude"],
    y = clarity_subset["latitude"]
)

# Set coordinate reference system 
clarity_subset = gpd.GeoDataFrame(clarity_subset).set_crs(epsg=epsg)

# Apply rounding function to the lat/lon columns of our subset clarity data
clarity_subset['latitude'] = clarity_subset['latitude'].apply(lambda x: round_002(x))
clarity_subset['longitude'] = clarity_subset['longitude'].apply(lambda x: round_002(x))

In [4]:

# Group by time and lat/lon. This will put multiple stations at the same lat/lon coordinate, and aggregate their data together.
# The 'freq' option in pd.Grouper can be changed to 'h' for doing diel cycle avgs over the whole study period
clarity_subset_hour = (
    clarity_subset.set_index('datetime').groupby([pd.Grouper('datetime', freq='1h'), 'latitude', 'longitude'])
    .agg( 
        sfc_no2_mean=('no2ConcIndividual.value', 'mean'),
        sfc_no2_median=('no2ConcIndividual.value', 'median'),
        sfc_no2_var=('no2ConcIndividual.value', 'var'),
        sfc_no2_sd=('no2ConcIndividual.value', 'std'),
        sensor_count=('datasourceid', 'nunique')
        ).reset_index()
)

clarity_subset_hour.head()

,datetime,latitude,longitude,sfc_no2_mean,sfc_no2_median,sfc_no2_var,sfc_no2_sd,sensor_count
0,2025-09-01,41.65,-87.55,9.031429,9.48,3.246748,1.801873,2
1,2025-09-01,41.67,-87.63,13.340000,13.16,7.046800,2.654581,1
2,2025-09-01,41.67,-87.61,7.200000,7.02,0.372400,0.610246,1
3,2025-09-01,41.67,-87.59,10.768333,10.99,0.937697,0.968347,2
4,2025-09-01,41.67,-87.55,11.107143,11.46,3.213057,1.792500,2


In [5]:
# Calculate the hourly vars of interest for tempo per pixel

tempo_hour_mean = tempo_raster.column_no2.resample(time='1h').mean(dim='time')
tempo_hour_median = tempo_raster.column_no2.resample(time='1h').median(dim='time')
tempo_hour_sd = tempo_raster.column_no2.resample(time='1h').std(dim='time')
tempo_hour_var = tempo_raster.column_no2.resample(time='1h').var(dim='time')

# Combine the above into a single dataframe.
tempo_hour_stats = xr.merge([
    tempo_hour_mean.rename('column_no2_mean'),
    tempo_hour_median.rename('column_no2_median'),
    tempo_hour_sd.rename('column_no2_sd'),
    tempo_hour_var.rename('column_no2_var')
])

# Note: The above can likely be combined to avoid creating four seperate datasets before merging.

/home/nlring/.conda/envs/chi-haq/lib/python3.14/site-packages/numpy/lib/_nanfunctions_impl.py:2015: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/nlring/.conda/envs/chi-haq/lib/python3.14/site-packages/numpy/lib/_nanfunctions_impl.py:2015: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/nlring/.conda/envs/chi-haq/lib/python3.14/site-packages/numpy/lib/_nanfunctions_impl.py:2015: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/nlring/.conda/envs/chi-haq/lib/python3.14/site-packages/numpy/lib/_nanfunctions_impl.py:2015: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/nlring/.conda/envs/chi-haq/lib/python3.14/site-packages/numpy/lib/_nanfunctions_impl.py:2015: RuntimeWarning: Degrees of freedom <= 0 for slic

In [6]:
# Rename the clarity datetime column to match the tempo time column
clarity_tempo = clarity_subset_hour.rename(columns={'datetime': 'time'})
# Define the index as spatial and temporal prior to converting to xarray
clarity_tempo = clarity_tempo.set_index(['time', 'latitude', 'longitude']).to_xarray()

# Normalize coord precision before merging so matching is consistent.
# This step is kind of hacky but necessary to ensure coordinates match exactly between clarity and tempo
# If there is a better way to do this, then please replace this. Without this step you get NaN values when combining.
clarity_tempo = clarity_tempo.assign_coords(
    latitude=((clarity_tempo.latitude.astype('float32') * 100).astype('int32') / 100).astype('float32'),
    longitude=((clarity_tempo.longitude.astype('float32') * 100).astype('int32') / 100).astype('float32'),
)

tempo_hour_stats = tempo_hour_stats.assign_coords(
    latitude=((tempo_hour_stats.latitude.astype('float32') * 100).astype('int32') / 100).astype('float32'),
    longitude=((tempo_hour_stats.longitude.astype('float32') * 100).astype('int32') / 100).astype('float32'),
)

# Here, we right join the clarity dataset to the tempo dataset.
# This keeps all points in tempo, but only matching points from clarity 
clarity_tempo = xr.merge([clarity_tempo, tempo_hour_stats], join='right', compat='override')

clarity_tempo

output_path = "~/Documents/01_Research/larc-chicago-haq/datasets/clarity-tempo-hourly.nc"
clarity_tempo.to_netcdf(output_path)